In [1]:
import pandas as pd
import numpy as np
import datetime
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
end_date = pd.to_datetime('2025-12-31 23:59:59')
start_date = pd.to_datetime('2025-01-01')
# end_date = pd.to_datetime(datetime.datetime.today().date())
# start_date = pd.to_datetime(datetime.datetime.today().date() - datetime.timedelta(days=13))
today = datetime.datetime.today().year*10000 + datetime.datetime.today().month*100 + datetime.datetime.today().day

print(f'今日是：{today}')
print(f'start_date：{start_date}')
print(f'end_date：{end_date}')

今日是：20260209
start_date：2025-01-01 00:00:00
end_date：2025-12-31 23:59:59


##### 读取数据，数据清洗(日期格式、'零件号'转换成字符串)，筛选本期数据

In [3]:
mass =  pd.read_excel('./eps3_mass.xlsx')
mass = mass[mass['状态'].isin(['已发布','已确认','已审批'])]
mass['生效日期'] = pd.to_datetime(mass['生效日期'].str[:10])
mass['失效日期'] = pd.to_datetime(mass['失效日期'].str[:10])
mass['创建时间'] = pd.to_datetime(mass['创建时间'].str[:10])
mass['审批通过时间'] = mass['审批通过时间'].astype(str)
mass['审批通过时间'] = pd.to_datetime(mass['审批通过时间'].str[:10])
mass['零件号'] = mass['零件号'].astype(str)
mass['供应商编码'] = mass['供应商编码'].astype(str)

In [4]:
usefull_mass_column = ['价格通知书号','供应商编码','供应商名称','零件号','零件名称','零件裸价','生效日期','失效日期','地区','创建人','审批通过时间']
df_mass_useful = mass[usefull_mass_column]
df_mass_useful_currently = df_mass_useful[(df_mass_useful['审批通过时间']<=end_date)&(df_mass_useful['审批通过时间']>=start_date)]
df_mass_useful_currently.head(2)
print(f'当期新增零件数{df_mass_useful_currently.shape[0]}')
## 后续如运算可以从此开始

当期新增零件数52924


In [5]:
df_mass_useful_currently.head(2)

,价格通知书号,供应商编码,供应商名称,零件号,零件名称,零件裸价,生效日期,失效日期,地区,创建人,审批通过时间
4145,CJ2512310002,770129,东实（湖北）动力电池系统有限公司,Z100134360,动力电池,18644.61,2025-12-01,2025-12-31,Xiangyang/,彭琴,2025-12-31
4146,CJ2512310002,770129,东实（湖北）动力电池系统有限公司,Z100083960,动力电池,18644.61,2025-12-01,2025-12-31,Xiangyang/,彭琴,2025-12-31


In [6]:
def expland_monthly(df):
    '价格履历表扩展成每个月'
    date_months = pd.date_range(
        start=df['生效日期'],
        end=df['失效日期'],
        freq='ME'
    )
    return date_months.to_period('M')

def expland_areas(df):
    """
    "地区"字段是"Wuhan/Xiangyang/"拆分为"Wuhan/","Xiangyang/"两行
    """
    area = df['地区'].str.rstrip('/').str.split('/')
    return area

In [7]:
df_part_currently_suppliers_nos = (
    df_mass_useful_currently.drop_duplicates(['供应商编码','零件号'])
    [['供应商编码','零件号']]
)
# df_part_currently_suppliers_nos：本期输机记录的供应商-零件号

In [9]:
df_price_difference_area_parts = pd.DataFrame()

for _, combox in df_part_currently_suppliers_nos.iterrows():
    supplier = combox['供应商编码']
    part_no = combox['零件号']

    # df_mass_useful_part_in_current:近期输机的某个零件(同一供应商)全部价格价格履历
    df_mass_useful_part_in_current = df_mass_useful[(df_mass_useful['零件号']==part_no) & (df_mass_useful['供应商编码']==supplier)]                                               
    if len(df_mass_useful_part_in_current) == 0:
        continue


    # 找出本月输机记录的生效月份
    months_currently = (
        df_mass_useful_currently[(df_mass_useful_currently['零件号']==part_no) & (df_mass_useful_currently['供应商编码']==supplier)]
        .apply(expland_monthly,axis=1)
        .explode()
        .drop_duplicates()
    )
    
    # 当期涉及的颜色件历史价格履历表扩展成每个月
    df_mass_useful_part_in_current['month'] = df_mass_useful_part_in_current.apply(expland_monthly,axis=1)
    df_mass_useful_part_in_current['area'] = expland_areas(df_mass_useful_part_in_current)
    
    df_mass_useful_colour_in_current_monthly_area = (
        df_mass_useful_part_in_current
        .explode('month')
        .explode('area')
            .sort_values(by=['零件号','month','area','审批通过时间'],ascending=False)
            .drop_duplicates(subset=['零件号','供应商编码','month','area'],keep='first')
        )
    # df_mass_useful_colour_in_current_monthly_area：近期输机的某个零件的全部月度、全部地区的价格记录
    
    recent_combinations = df_mass_useful_colour_in_current_monthly_area[['供应商编码', 'month']].drop_duplicates()
    df_price_difference_area_part = pd.DataFrame()
    
    for month in months_currently:
        all_parts_records = df_mass_useful_colour_in_current_monthly_area[
            (df_mass_useful_colour_in_current_monthly_area['month'] == month)
        ]
        if len(all_parts_records) > 1:
            unique_price = all_parts_records['零件裸价'].unique()
            if unique_price.size > 1:
                df_price_difference_area_part = pd.concat([df_price_difference_area_part,all_parts_records],ignore_index=True)
        
    if len(df_price_difference_area_part) > 0:
        df_price_difference_area_parts = pd.concat([df_price_difference_area_parts,df_price_difference_area_part],ignore_index=True)

if len(df_price_difference_area_parts) > 0:
    df_price_difference_area_parts
else:
    print('本期不同区域输机未发现价格差异')

ValueError: Neither `start` nor `end` can be NaT

In [ ]:
# 输出结果
if len(df_price_difference_area_parts) > 0:
    useful_columns_export = ['价格通知书号','供应商编码','供应商名称','零件号','零件名称','零件裸价','生效日期','失效日期','地区','创建人']

    df_price_difference_colours_export = (
    df_price_difference_area_parts[useful_columns_export]
    .drop_duplicates(['价格通知书号','零件号'])
    .sort_values(['零件号','价格通知书号'],ascending=False)
    )

    df_price_difference_colours_export.to_excel(f'./同件不同价清单/本期同件不同价_地区_2025_{today}.xlsx',index=False)

## 循环内的测试

In [ ]:
# supplier = 100288
# part_no = 'Z100164960'

# # df_mass_useful_part_in_current:近期输机的某个零件(同一供应商)全部价格价格履历
# df_mass_useful_part_in_current = df_mass_useful[(df_mass_useful['零件号']==part_no) & (df_mass_useful['供应商编码']==supplier)]                                               

# # 找出本月输机记录的生效月份
# months_currently = (
#     df_mass_useful_currently[(df_mass_useful_currently['零件号']==part_no) & (df_mass_useful_currently['供应商编码']==supplier)]
#     .apply(expland_monthly,axis=1)
#     .explode()
#     .drop_duplicates()
# )

# # 当期涉及的颜色件历史价格履历表扩展成每个月
# df_mass_useful_part_in_current['month'] = df_mass_useful_part_in_current.apply(expland_monthly,axis=1)
# df_mass_useful_part_in_current['area'] = expland_areas(df_mass_useful_part_in_current)

# df_mass_useful_colour_in_current_monthly_area = (
#     df_mass_useful_part_in_current
#     .explode('month')
#     .explode('area')
#         .sort_values(by=['零件号','month','area','审批通过时间'],ascending=False)
#         .drop_duplicates(subset=['零件号','供应商编码','month','area'],keep='first')
#     )
# # df_mass_useful_colour_in_current_monthly_area：近期输机的某个零件的全部月度、全部地区的价格记录

# recent_combinations = df_mass_useful_colour_in_current_monthly_area[['供应商编码', 'month']].drop_duplicates()
# df_price_difference_area_part = pd.DataFrame()
# for month in months_currently:
#     all_parts_records = df_mass_useful_colour_in_current_monthly_area[
#         (df_mass_useful_colour_in_current_monthly_area['month'] == month)
#     ]
#     if len(all_parts_records) > 1:
#         unique_price = all_parts_records['零件裸价'].unique()
#         if unique_price.size > 1:
#             df_price_difference_area_part = pd.concat([df_price_difference_area_part,all_parts_records],ignore_index=True)


# df_price_difference_area_part